# Week 46, Day 3: Balance: AVL rotations

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w046d3.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
class Node:
    def __init__(self, val):
        self.val, self.left, self.right, self.height = val, None, None, 0

def height(n):
    return n.height if n else -1

def rotate_left(x):
    y = x.right
    x.right = y.left
    y.left = x
    x.height = 1 + max(height(x.left), height(x.right))
    y.height = 1 + max(height(y.left), height(y.right))
    return y

def preorder(n):
    return [] if n is None else [n.val] + preorder(n.left) + preorder(n.right)

a, b, c = Node(1), Node(2), Node(3)
a.right, b.right = b, c
b.height, a.height = 1, 2
root = rotate_left(a)
print(preorder(root), root.height)

In [ ]:
class Node:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right
        self.height = 1 + max(height(left), height(right))

def height(n):
    return n.height if n else -1

def update(n):
    n.height = 1 + max(height(n.left), height(n.right))

def rotate_right(y):
    x = y.left
    y.left = x.right
    x.right = y
    update(y)              # y is now below x: update it first
    update(x)
    return x               # the new root of this subtree

def inorder(n):
    return [] if n is None else inorder(n.left) + [n.val] + inorder(n.right)

root = Node(30, Node(20, Node(10), Node(25)), Node(40))
print(inorder(root), "root", root.val, "height", root.height)
root = rotate_right(root)
print(inorder(root), "root", root.val, "height", root.height, "right child", root.right.val)

In [ ]:
class Node:
    def __init__(self, val):
        self.val, self.left, self.right, self.height = val, None, None, 0

def height(n):
    return n.height if n else -1

def update(n):
    n.height = 1 + max(height(n.left), height(n.right))

def balance(n):
    return height(n.left) - height(n.right)

def rotate_right(y):
    x = y.left
    y.left, x.right = x.right, y
    update(y)
    update(x)
    return x

def rotate_left(x):
    y = x.right
    x.right, y.left = y.left, x
    update(x)
    update(y)
    return y

def rebalance(n):
    update(n)
    if balance(n) > 1:
        if balance(n.left) < 0:
            n.left = rotate_left(n.left)      # left-right: straighten first
        return rotate_right(n)
    if balance(n) < -1:
        if balance(n.right) > 0:
            n.right = rotate_right(n.right)   # right-left: straighten first
        return rotate_left(n)
    return n

def insert(n, val):
    if n is None:
        return Node(val)
    if val < n.val:
        n.left = insert(n.left, val)
    elif val > n.val:
        n.right = insert(n.right, val)
    else:
        return n
    return rebalance(n)

def preorder(n):
    return [] if n is None else [n.val] + preorder(n.left) + preorder(n.right)

root = None
for v in [10, 20, 30, 40, 50, 25]:
    root = insert(root, v)
    print(f"insert {v}: {preorder(root)}")

In [ ]:
class Node:
    def __init__(self, val):
        self.val, self.left, self.right, self.height = val, None, None, 0

def height(n):
    return n.height if n else -1

def update(n):
    n.height = 1 + max(height(n.left), height(n.right))

def balance(n):
    return height(n.left) - height(n.right)

def rotate_right(y):
    x = y.left
    y.left, x.right = x.right, y
    update(y)
    update(x)
    return x

def rotate_left(x):
    y = x.right
    x.right, y.left = y.left, x
    update(x)
    update(y)
    return y

def rebalance(n):
    update(n)
    if balance(n) > 1:
        if balance(n.left) < 0:
            n.left = rotate_left(n.left)
        return rotate_right(n)
    if balance(n) < -1:
        if balance(n.right) > 0:
            n.right = rotate_right(n.right)
        return rotate_left(n)
    return n

def insert(n, val):
    if n is None:
        return Node(val)
    if val < n.val:
        n.left = insert(n.left, val)
    elif val > n.val:
        n.right = insert(n.right, val)
    else:
        return n
    return rebalance(n)

import random
for name, order in [("sorted", list(range(1, 1024))), ("shuffled", random.Random(1).sample(range(1, 1024), 1023))]:
    root = None
    for v in order:
        root = insert(root, v)
    print(f"{name:<9} height {root.height}")

> **Key idea.** An AVL tree keeps every node's two subtree heights within 1 of each other, storing a height in each node (−1 for empty). A rotation changes three links and keeps the in-order sequence, moving height from one side to the other; update the lower node's height first. Balance factor +2 with a left child leaning left (or level) needs one right rotation; leaning right (a zigzag) needs a left rotation at the child first. −2 is the mirror image. AVL insert and delete are the recursive BST versions with `return rebalance(node)`. The fewest nodes at height h follow a Fibonacci-like recurrence, so the height is at most about 1.44 log₂ n, and every operation is O(log n).

### Exercise: Is it an AVL tree?

The nodes here have no stored heights. Write `heights(root)`, returning a dict from each node's value to its height (leaves 0), computed in **one** post-order pass, and `is_avl(root)`: a BST (bounds passed down) where every balance factor is −1, 0 or 1.

In [ ]:
class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def heights(root):
    ___

def is_avl(root):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("T = TreeNode\ndef test_avl():\n    \"checking\"\n    good = T(8, T(4, T(2), T(6)), T(9))\n    assert heights(good) == {8: 2, 4: 1, 2: 0, 6: 0, 9: 0} and heights(None) == {}\n    assert is_avl(good) and is_avl(None) and is_avl(T(1, None, T(2)))\n    assert not is_avl(T(8, T(4, T(2)))), \"8 is left heavy by 2\"\n    assert not is_avl(T(8, T(4, T(2), T(9)), T(10))), \"9 is not less than 8\"\n    deep = T(50, T(25, T(10, T(5)), T(30)), T(75, None, T(80)))\n    assert is_avl(deep) and not is_avl(T(50, T(25, T(10, T(5)), T(30)), T(75)))")

### Exercise: Rotations

Write `rotate_left(x)` and `rotate_right(y)`: perform the rotation, update the two nodes' stored heights in the right order, and return the new subtree root.

In [ ]:
class Node:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right
        self.height = 1 + max(height(left), height(right))

def height(n):
    return n.height if n else -1

def rotate_left(x):
    ___

def rotate_right(y):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def pre(n):\n    return [] if n is None else [(n.val, n.height)] + pre(n.left) + pre(n.right)\ndef test_rotate():\n    \"rotations\"\n    N = Node\n    r = rotate_right(N(30, N(20, N(10), N(25)), N(40)))\n    assert pre(r) == [(20, 2), (10, 0), (30, 1), (25, 0), (40, 0)]\n    r = rotate_left(N(1, None, N(2, None, N(3))))\n    assert pre(r) == [(2, 1), (1, 0), (3, 0)]\n    r = rotate_left(N(10, N(5), N(20, N(15), N(30, None, N(40)))))\n    assert pre(r) == [(20, 2), (10, 1), (5, 0), (15, 0), (30, 1), (40, 0)]\n    assert pre(rotate_right(rotate_left(N(2, N(1), N(3))))) == [(2, 1), (1, 0), (3, 0)]")

### Exercise: The four cases

Write `rebalance(n)`: update `n`'s height, then apply whichever of the four fixes is needed (or none), returning the subtree's new root. Also write `case(n)`, returning `"LL"`, `"LR"`, `"RR"`, `"RL"` or `None` for the case `n` is in (after updating its height). Rotations are provided.

In [ ]:
class Node:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right
        self.height = 1 + max(height(left), height(right))

def height(n):
    return n.height if n else -1

def update(n):
    n.height = 1 + max(height(n.left), height(n.right))

def balance(n):
    return height(n.left) - height(n.right)

def rotate_right(y):
    x = y.left
    y.left, x.right = x.right, y
    update(y)
    update(x)
    return x

def rotate_left(x):
    y = x.right
    x.right, y.left = y.left, x
    update(x)
    update(y)
    return y

def case(n):
    ___

def rebalance(n):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("N = Node\ndef pre(n):\n    return [] if n is None else [n.val] + pre(n.left) + pre(n.right)\ndef test_cases():\n    \"the four cases\"\n    trees = {\n        \"LL\": (lambda: N(3, N(2, N(1))), [2, 1, 3]),\n        \"LR\": (lambda: N(3, N(1, None, N(2))), [2, 1, 3]),\n        \"RR\": (lambda: N(1, None, N(2, None, N(3))), [2, 1, 3]),\n        \"RL\": (lambda: N(1, None, N(3, N(2))), [2, 1, 3]),\n        None: (lambda: N(2, N(1)), [2, 1]),\n    }\n    for name, (make, want) in trees.items():\n        assert case(make()) == name, name\n        r = rebalance(make())\n        assert pre(r) == want and r.height == 1, name\n    r = rebalance(N(50, N(30, N(20, N(10)), N(40)), N(60)))\n    assert pre(r) == [30, 20, 10, 50, 40, 60] and r.height == 2\n    r = rebalance(N(50, N(30, N(20), N(40, N(35))), N(60)))\n    assert pre(r) == [40, 30, 20, 35, 50, 60] and r.height == 2")

### Exercise: An AVL set

Write `AVLSet` with `add`, `discard`, `__contains__`, `__len__`, `to_list()` (sorted) and `height()` (−1 when empty), rebalancing after every insert and delete. The checks insert 20,000 values in increasing order (the tree must stay at most 1.44 log₂ n tall) and run thousands of random operations, checking the AVL property throughout.

In [ ]:
class Node:
    def __init__(self, val):
        self.val, self.left, self.right, self.height = val, None, None, 0

class AVLSet:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random, math\ndef check(n, lo=float(\"-inf\"), hi=float(\"inf\")):\n    if n is None:\n        return -1\n    assert lo < n.val < hi, \"order\"\n    a, b = check(n.left, lo, n.val), check(n.right, n.val, hi)\n    assert abs(a - b) <= 1, f\"unbalanced at {n.val}\"\n    assert n.height == 1 + max(a, b), f\"stale height at {n.val}\"\n    return n.height\ndef root_of(s):\n    roots = [v for v in vars(s).values() if v is None or isinstance(v, Node)]\n    assert len(roots) == 1, \"keep one root attribute\"\n    return roots[0]\ndef test_sorted():\n    \"sorted input\"\n    s = AVLSet()\n    for v in range(20_000):\n        s.add(v)\n    assert len(s) == 20_000 and s.height() <= 1.44 * math.log2(20_000) and 0 in s and 20_000 not in s\n    check(root_of(s))\n    for v in range(0, 20_000, 2):\n        s.discard(v)\n    check(root_of(s))\n    assert len(s) == 10_000 and s.to_list()[:3] == [1, 3, 5]\ndef test_random():\n    \"random operations\"\n    rng = random.Random(10)\n    s, ref = AVLSet(), set()\n    for i in range(4000):\n        v = rng.randrange(300)\n        if rng.random() < 0.55:\n            s.add(v)\n            ref.add(v)\n        else:\n            s.discard(v)\n            ref.discard(v)\n        assert len(s) == len(ref)\n        if i % 400 == 0:\n            check(root_of(s))\n    assert s.to_list() == sorted(ref) and AVLSet().height() == -1")

### Exercise: The sparsest AVL trees

Write `min_nodes(h)`: the fewest nodes an AVL tree of height h can have (`min_nodes(0) == 1`, `min_nodes(1) == 2`), with a loop. Then `max_height(n)`: the greatest height an AVL tree with n ≥ 1 nodes can have (the largest h with `min_nodes(h) <= n`). Finally `build_sparse(h)`: an actual AVL tree of height h with exactly `min_nodes(h)` nodes holding the values 1, 2, 3, ... (any valid shape). Build it by making the two subtrees with sizes `min_nodes(h - 1)` and `min_nodes(h - 2)` and numbering values in order.

In [ ]:
class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def min_nodes(h):
    ___

def max_height(n):
    ___

def build_sparse(h):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_sparse():\n    \"sparse trees\"\n    assert [min_nodes(h) for h in range(10)] == [1, 2, 4, 7, 12, 20, 33, 54, 88, 143]\n    assert [max_height(n) for n in [1, 2, 3, 4, 6, 7, 1_000_000]] == [0, 1, 1, 2, 2, 3, 27]\n    def info(n, lo, hi):\n        if n is None:\n            return -1, 0\n        assert lo < n.val < hi\n        (a, ca), (b, cb) = info(n.left, lo, n.val), info(n.right, n.val, hi)\n        assert abs(a - b) <= 1\n        return 1 + max(a, b), ca + cb + 1\n    for h in range(8):\n        t = build_sparse(h)\n        assert info(t, 0, 10**9) == (h, min_nodes(h)), h")

### Bug hunt: Heights in the wrong order

This right rotation updates the new root's height **before** the old root's, so it uses a stale height and gets it wrong: this prints `3 0`. Update the lower node first so it prints `1 0`.

In [ ]:
class Node:
    def __init__(self, val, left=None):
        self.val, self.left, self.right = val, left, None
        self.height = 1 + max(height(left), -1)

def height(n):
    return n.height if n else -1

def update(n):
    n.height = 1 + max(height(n.left), height(n.right))

def rotate_right(y):
    x = y.left
    y.left = x.right
    x.right = y
    update(x)
    update(y)
    return x

root = rotate_right(Node(3, Node(2, Node(1))))
print(root.height, root.right.height)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"lower first\"\n    expect_output(\"1 0\")")

### Bug hunt: One rotation for a zigzag

This rebalance always does a single rotation, so a left-right zigzag just turns into a right-left one and stays unbalanced: inserting 3, 1, 2 prints `[1, 3, 2]`. Handle the zigzag cases with a rotation at the child first, so it prints `[2, 1, 3]`.

In [ ]:
class Node:
    def __init__(self, val):
        self.val, self.left, self.right, self.height = val, None, None, 0

def height(n):
    return n.height if n else -1

def update(n):
    n.height = 1 + max(height(n.left), height(n.right))

def balance(n):
    return height(n.left) - height(n.right)

def rotate_right(y):
    x = y.left
    y.left, x.right = x.right, y
    update(y)
    update(x)
    return x

def rotate_left(x):
    y = x.right
    x.right, y.left = y.left, x
    update(x)
    update(y)
    return y

def rebalance(n):
    update(n)
    if balance(n) > 1:
        return rotate_right(n)
    if balance(n) < -1:
        return rotate_left(n)
    return n

def insert(n, val):
    if n is None:
        return Node(val)
    if val < n.val:
        n.left = insert(n.left, val)
    else:
        n.right = insert(n.right, val)
    return rebalance(n)

def preorder(n):
    return [] if n is None else [n.val] + preorder(n.left) + preorder(n.right)

root = None
for v in [3, 1, 2]:
    root = insert(root, v)
print(preorder(root))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"zigzags\"\n    expect_output(\"[2, 1, 3]\")\n    r = None\n    for v in [1, 3, 2]:\n        r = insert(r, v)\n    assert preorder(r) == [2, 1, 3]")

### Bug hunt: Never noticing

This insert rebalances but never updates heights on the way up, so every node still thinks it's a leaf, no imbalance is ever seen, and inserting 1 to 7 in order builds a chain: this prints `6`. Update each node's height before checking its balance, so the tree stays short and it prints `2`.

In [ ]:
class Node:
    def __init__(self, val):
        self.val, self.left, self.right, self.height = val, None, None, 0

def height(n):
    return n.height if n else -1

def update(n):
    n.height = 1 + max(height(n.left), height(n.right))

def balance(n):
    return height(n.left) - height(n.right)

def rotate_right(y):
    x = y.left
    y.left, x.right = x.right, y
    update(y)
    update(x)
    return x

def rotate_left(x):
    y = x.right
    x.right, y.left = y.left, x
    update(x)
    update(y)
    return y

def rebalance(n):
    if balance(n) > 1:
        if balance(n.left) < 0:
            n.left = rotate_left(n.left)
        return rotate_right(n)
    if balance(n) < -1:
        if balance(n.right) > 0:
            n.right = rotate_right(n.right)
        return rotate_left(n)
    return n

def insert(n, val):
    if n is None:
        return Node(val)
    if val < n.val:
        n.left = insert(n.left, val)
    else:
        n.right = insert(n.right, val)
    return rebalance(n)

def true_height(n):
    return -1 if n is None else 1 + max(true_height(n.left), true_height(n.right))

root = None
for v in range(1, 8):
    root = insert(root, v)
print(true_height(root))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"update first\"\n    expect_output(\"2\")")

### Puzzle: A left rotation

Arrange the lines into a left rotation that keeps stored heights correct. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
# Insert into an empty AVL tree, in order: 10, 20, 30, 40, 50, 25.
# (The section "The four cases" printed the steps.)

p = ___   # the root after inserting 10, 20, 30
q = ___   # the case fixed when 25 is inserted: "LL", "LR", "RR" or "RL"
r = ___   # the final pre-order traversal
s = ___   # the final height
t = ___   # the fewest nodes an AVL tree of height 4 can have

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == 20 and q == \"RL\" and r == [30, 20, 10, 25, 40, 50], \"p, q and r\"\n    assert s == 2 and t == 12, \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a balanced ranked map

Combine Day 2's challenge with today: write `RankedMap`, an AVL tree of key and value pairs where every node also stores its **subtree size**, so it supports:

- `m[key] = value` (insert or update), `m[key]` (`KeyError`), `del m[key]` (`KeyError`), `len(m)`, `key in m`;
- `rank(key)` (how many keys are less than `key`) and `select(k)` (the key with k smaller keys; `IndexError` out of range);
- `height()`.

Rotations must update sizes as well as heights. The checks insert 30,000 keys in increasing order (height at most 1.44 log₂ n), then run thousands of random operations against a dict and a sorted list.

In [ ]:
class Node:
    def __init__(self, key, value):
        self.key, self.value = key, value
        self.left = self.right = None
        self.height, self.size = 0, 1

class RankedMap:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random, bisect, math\ndef test_sorted():\n    \"sorted input stays short\"\n    m = RankedMap()\n    for k in range(30_000):\n        m[k] = k * k\n    assert len(m) == 30_000 and m.height() <= 1.44 * math.log2(30_000) and m[123] == 15129\n    assert m.select(0) == 0 and m.select(29_999) == 29_999 and m.rank(1000) == 1000\n    for k in range(0, 30_000, 3):\n        del m[k]\n    assert len(m) == 20_000 and m.select(0) == 1 and m.height() <= 1.44 * math.log2(20_000)\ndef test_random():\n    \"against a dict\"\n    rng = random.Random(11)\n    m, ref = RankedMap(), {}\n    for i in range(5000):\n        k = rng.randrange(400)\n        if rng.random() < 0.6:\n            m[k] = i\n            ref[k] = i\n        elif k in ref:\n            del m[k]\n            del ref[k]\n        else:\n            try:\n                del m[k]\n                assert False\n            except KeyError:\n                pass\n        assert len(m) == len(ref)\n    keys = sorted(ref)\n    for k in range(-1, 402, 5):\n        assert m.rank(k) == bisect.bisect_left(keys, k) and (k in m) == (k in ref)\n    for i in range(0, len(keys), 3):\n        assert m[m.select(i)] == ref[keys[i]]\n    try:\n        m.select(len(keys))\n        assert False\n    except IndexError:\n        pass")

## Warm-up

A chain 1, 2, 3 of height 2 became a tree with 2 at the root and height 1. Three pointer changes, no values moved, and the in-order traversal is still 1, 2, 3. That's a **rotation**, and it's the only tool an AVL tree needs.

## What balanced means

A perfectly balanced tree (every level full) is too strict to maintain cheaply: adding one value could force the whole tree to be rebuilt. AVL trees use a looser rule that's still strong enough:

> At **every** node, the heights of the left and right subtrees differ by **at most 1**.

Each node stores its own **height** (edges to its deepest leaf; a leaf is 0, and an empty subtree counts as −1). The **balance factor** is `height(left) − height(right)`, and the rule says it must be −1, 0 or 1 everywhere.

```text
  balanced (AVL):                 not AVL:
          8   bf 0                      8   bf +2   (left height 1, right −1)
         / \                           /
        4   9                         4
       / \                           /
      2   6                         2
```

After an insert or delete, only nodes on the path from the change up to the root can have changed height, so only they need checking, and that's O(log n) nodes.

## Rotations

A **right rotation** at `y` lifts its left child `x` into `y`'s place; `y` becomes `x`'s right child, and `x`'s old right subtree `B` moves across to become `y`'s left:

```text
      right rotation at y               left rotation at x
            y                  x            x                    y
           / \                / \          / \                  / \
          x   C     ->       A   y        A   y       ->       x   C
         / \                    / \          / \              / \
        A   B                  B   C        B   C            A   B
```

In both pictures the in-order sequence is `A x B y C`: every value in `B` was between `x` and `y` before, and it still is. So a rotation **keeps the BST property** while making one side shorter and the other taller. It changes three links, then recomputes the heights of the two nodes involved, the lower one (`y` for a right rotation) **first**, since the upper one's height depends on it.

## The four cases

After inserting or deleting below a node, its balance factor can reach +2 or −2. Look at the **taller child** to decide what to do:

| Node's balance | Taller child's balance | Case | Fix |
|---|---|---|---|
| +2 (left heavy) | ≥ 0 | left-left | rotate right at the node |
| +2 | < 0 | left-right | rotate left at the left child, then right at the node |
| −2 (right heavy) | ≤ 0 | right-right | rotate left at the node |
| −2 | > 0 | right-left | rotate right at the right child, then left at the node |

The "straight line" cases (left-left, right-right) take one rotation. The "zigzag" cases need two: the first straightens the zigzag into a line, the second fixes the line. A single rotation on a zigzag just makes the opposite zigzag.

```text
  left-right: insert 3, 1, 2
      3  bf +2          3              2
     /                 /              / \
    1    bf −1  ->    2      ->      1   3
     \               /
      2             1
         rotate left at 1   rotate right at 3
```

Watch the steps: 30 caused a right-right fix at 10; 50 a right-right fix at 30; and 25 a right-left fix at the root.

## AVL insert and delete

That example **is** AVL insertion: yesterday's recursive insert, with `return rebalance(n)` instead of `return n`. Because each call returns its (possibly new) subtree root, a rotation anywhere is automatically linked back into the parent. After an insert, at most one rebalance actually rotates (the fixed subtree goes back to its old height, so nothing above it changes), but checking every node on the path costs only O(log n) anyway.

**Delete** is yesterday's delete with the same change: every `return node` that follows a recursive call becomes `return rebalance(node)`. A delete can need rotations at several levels on the way up, still O(log n) in total.

Compare Day 1: 1,022 and 23 for a plain BST. Sorted input into an AVL tree even produces a perfect tree here.

## How short is short?

How tall can an AVL tree with n nodes get? Turn it round: what's the **fewest** nodes an AVL tree of height h can have? Such a tree has a root, one subtree of height h − 1 and (to be as sparse as possible) the other of height h − 2, each as sparse as possible too:

N(h) = N(h − 1) + N(h − 2) + 1, with N(0) = 1 and N(1) = 2.

That's 1, 2, 4, 7, 12, 20, 33, 54, 88... which grows like the Fibonacci numbers, by a factor of about 1.618 per level. So n nodes allow a height of at most about log base 1.618 of n, which is about **1.44 log₂ n**. A million values: height at most 27, against 19 for a perfect tree and a million for the worst plain BST.

Other balanced trees make different trade-offs. **Red-black trees** (Guibas and Sedgewick, 1978, building on Rudolf Bayer's 1972 work) allow a little more imbalance (height up to 2 log₂ n) in exchange for fewer rotations on delete; they're behind Java's `TreeMap` and C++'s `std::map`. **B-trees** (Bayer and McCreight, 1970) give each node many children, which suits disks and databases. Python's standard library has no balanced tree at all: for many jobs, a sorted list with `bisect` (Week 18) is fast enough, because shifting elements in memory is quick until lists get very large.

## Practice

## Quiz

## Daily challenge

## Before you go

On paper, insert 1 to 7 in order into an AVL tree, drawing the tree after each insertion and naming each rotation. You should end with a perfect tree. Tomorrow: more tree problems, using the BST property to answer questions about ancestors, paths and pairs of trees.